# 1. Contextual Bandits for Re-engagement Notifications



## Scenario

A user has gone silent for $N$ weeks.  We send **one** re-engagement push notification (or email) to bring them back.  The system must decide **which message variant** to send, using what it knows about the user.

## Formal setup

| Component | Symbol | Description |
|-----------|--------|-------------|
| **Context** | $x$ | Features describing the dormant user (e.g. `lifetime_purchases`, `days_as_customer`) |
| **Action** | $a \in \{1,\dots,K\}$ | Which message variant to send |
| **Reward** | $r \in \{0, 1\}$ | Whether the user clicks through (1) or ignores (0) |

At each round the system observes a context $x$, chooses an action $a$, and receives a **bandit reward**: the outcome for the chosen action only.  We never see what *would have happened* under a different action (the fundamental **counterfactual** problem).

## Ground-truth click model (unknown to the learner)

Each (user, message) pair has an independent click probability:

$$
p(\text{click} \mid x, a) \;=\; \sigma\!\bigl(W_{\text{TRUE}}[a] \cdot x\bigr)
$$

where $\sigma(z) = \frac{1}{1 + e^{-z}}$ is the sigmoid function, and $x = [1, \;\text{lifetime\_purchases\_std}, \;\text{days\_as\_customer\_std}]$.

These probabilities are **independent Bernoulli means** — they do **not** sum to 1 across actions.  They represent "how likely is a click *if* we send this particular message to this particular user."

### Role of the bias term

The leading 1 in $x$ is a **bias/intercept** term.  Each row of $W_{\text{TRUE}}$ has the form $[\text{bias}, \; w_{\text{purchases}}, \; w_{\text{days}}]$, so the logit expands to:

$$
z_a(x) = \underbrace{W[a, 0]}_{\text{bias}} \cdot 1 \;+\; W[a, 1] \cdot \text{purchases\_std} \;+\; W[a, 2] \cdot \text{days\_std}
$$

When both features are at their population mean (standardized = 0), the feature terms vanish and we're left with $p(\text{click}) = \sigma(\text{bias})$.  This gives each action its **baseline CTR** — the click rate for a completely average user:

| Action | Bias | $\sigma$(bias) |
|--------|------|---------------|
| "See what's new" | −2.5 | **7.6%** |
| "Personalized picks" | −2.5 | **7.6%** |
| "We miss you" | −2.8 | **5.7%** |
| "Trending picks" | −2.2 | **10.0%** |

These baselines are realistic for re-engagement notifications (~6–10%).  The feature weights then push CTRs up or down from this baseline depending on the user's profile.

## Oracle decision boundaries

The **oracle policy** always picks the action with the highest true CTR:

$$
a^*(x) \;=\; \arg\max_{a} \; W_{\text{TRUE}}[a] \cdot x
$$

(We can drop the sigmoid because it's strictly monotonic — it preserves the ranking.)

The boundary between any two actions $a$ and $b$ is where their logits tie:

$$
W_{\text{TRUE}}[a] \cdot x \;=\; W_{\text{TRUE}}[b] \cdot x
\quad\Longleftrightarrow\quad
\bigl(W_{\text{TRUE}}[a] - W_{\text{TRUE}}[b]\bigr) \cdot x \;=\; 0
$$

This is a **linear equation** in $x$, so every pairwise boundary is a straight line.  The result is a partition of the feature space into convex regions — one per winning action.

> **The bandit's job** is to discover these regions through exploration and reward feedback, *without* ever knowing $W_{\text{TRUE}}$.  The gap between the oracle's CTR and the bandit's CTR is the **regret**.

## Reading the CTR profiles

Each curve shows how one message's click probability changes as we sweep a single feature, holding the other at its population mean.

**CTR vs. purchase history** (left panel)
- **"Personalized picks"** rises steeply — more purchase history gives the system better data to personalize with, so the message lands harder.
- **"See what's new"** rises more gently — frequent buyers are curious about new inventory, but the effect is weaker.
- **"We miss you"** *decreases* — emotional appeals don't resonate as well with transactional power shoppers.
- **"Trending picks"** decreases gently — social proof matters less to users who already know what they want.

**CTR vs. account age** (right panel)
- **"We miss you"** climbs steeply — the longer someone's been a customer, the more an emotional appeal feels genuine rather than generic.
- **"Personalized picks"** rises moderately — longer tenure means more data to personalize with.
- **"See what's new"** *drops* — a long-tenured user has probably seen most of what's new; this framing works better for recently-lapsed users.
- **"Trending picks"** drops gently — social proof becomes less compelling over time for established customers.

Baseline CTRs around the population mean sit at **~6–10%**, realistic for re-engagement notifications. The best oracle matches peak around **30–40%** for extreme users (rare, but plausible for the tail of the distribution).

# 2. The Bandit Learning Loop



Now we introduce the **learner** — a VowpalWabbit contextual-bandit agent that doesn't know $W_{\text{TRUE}}$ and must figure out which message works best for each user through trial and error.

### The interaction loop

On every round the system performs four steps:

1. **Observe context** — a dormant user's features arrive.
2. **Predict & explore** — the policy outputs a probability distribution over the $K$ actions and we **sample** an action from it.  The probability assigned to the chosen action is the **propensity** $\mu(a \mid x)$.
3. **Observe reward** — the environment reveals whether the user clicked (1) or not (0) *for the chosen action only* (bandit feedback — we never see outcomes for unchosen actions).
4. **Learn** — the (action, cost, propensity, features) tuple is fed back to VW so it can update its internal model.

### Exploration vs. exploitation

The policy's action distribution is controlled by $\varepsilon$-greedy exploration:

$$
\pi(a \mid x) = 
\begin{cases}
1 - \varepsilon + \dfrac{\varepsilon}{K} & \text{if } a = \arg\max_{a'} \hat{r}(x, a') \\[6pt]
\dfrac{\varepsilon}{K} & \text{otherwise}
\end{cases}
$$

where:
- $K$ is the number of actions (here $K = 4$, our four message variants)
- $\hat{r}(x, a')$ is VW's **estimated reward** for action $a'$ given context $x$ — its current best guess of the CTR, learned from past observations
- $\varepsilon$ controls the explore/exploit tradeoff

In words: with probability $\varepsilon$ we **explore** (pick uniformly at random), and with probability $1 - \varepsilon$ we **exploit** (pick the action with the highest estimated reward).  The formula folds both into a single distribution — the greedy action gets the exploitation mass *plus* its share of the exploration mass.

Concrete example with $K=4$, $\varepsilon=0.1$:
- Best action: $0.9 + 0.025 = 0.925$ (92.5%)
- Each other action: $0.025$ (2.5%)

Key settings:
- $\varepsilon = 1$ → **uniform random** (pure exploration).  Every action gets $\frac{1}{K} = 0.25$.
- $\varepsilon = 0$ → **greedy** (pure exploitation).  The model always picks its current best guess.
- In practice we start with $\varepsilon = 1$ during a **warm-up phase** to collect unbiased data, then reduce $\varepsilon$ to exploit what we've learned.

### Why propensity matters — $\mu$ vs. $\pi$

Two policies appear in this notebook and it's important to keep them straight:

| Symbol | Name | Role |
|--------|------|------|
| $\mu(a \mid x)$ | **Logging / behavior policy** | The policy that *actually chose the actions* when the data was collected.  During warm-up, $\mu$ is uniform ($\varepsilon = 1$), so $\mu(a \mid x) = \frac{1}{K} = 0.25$. |
| $\pi(a \mid x)$ | **Target / candidate policy** | A *different* policy we want to evaluate or deploy (e.g. $\varepsilon = 0.05$). |

The distinction matters for **off-policy evaluation (OPE)** — estimating how $\pi$ would have performed using only data collected under $\mu$.  The key tool is the **importance weight**:

$$
w_i = \frac{\pi(a_i \mid x_i)}{\mu(a_i \mid x_i)}
$$

If $\pi$ would have chosen action $a_i$ *more often* than $\mu$ did, we upweight that observation; if less often, we downweight it.  This corrects for the mismatch between what we did ($\mu$) and what we wish we had done ($\pi$).

Recording $\mu(a \mid x)$ with every observation during data collection is what makes this correction possible.

Two common OPE estimators built on importance weights:

**IPS (Inverse Propensity Scoring)** — an unbiased estimate of $\pi$'s expected reward:

$$
\hat{V}_{\text{IPS}}(\pi) = \frac{1}{N} \sum_{i=1}^{N} w_i \cdot r_i
$$

**SNIPS (Self-Normalized IPS)** — normalizes by the sum of weights, trading a small bias for much lower variance:

$$
\hat{V}_{\text{SNIPS}}(\pi) = \frac{\sum_{i=1}^{N} w_i \cdot r_i}{\sum_{i=1}^{N} w_i}
$$

In practice SNIPS tends to give more stable estimates, which is why we use it later to select the best $\varepsilon$.

### A note on $\hat{r}$ — VW's internal model

The $\hat{r}(x, a')$ in the $\varepsilon$-greedy formula is VW's **learned reward estimate**.  Internally VW maintains a weight vector $\hat{W}[a]$ for each action and predicts:

$$
\hat{r}(x, a) = \hat{W}[a] \cdot x
$$

This is a **raw linear prediction** — no sigmoid.  Our true environment uses $\sigma(W_{\text{TRUE}}[a] \cdot x)$, but VW doesn't know that.  It doesn't need to recover the exact click probabilities — it just needs to *rank* the actions correctly for each context, and a linear model can get the ranking right without the sigmoid since $\sigma$ is monotonic (preserves ordering).



# 3. Doubly Robust vs. IPS/SNIPS — where each is used



These are related ideas (all based on importance weighting) but applied at different stages:

| Method | When | Purpose |
|--------|------|---------|
| **Doubly Robust** (`--cb_type dr`) | During each online learning step | Helps VW estimate costs for *unchosen* actions to update its model |
| **IPS / SNIPS** | After warm-up (batch OPE) | Estimates a candidate policy's CTR from logged data to pick the best $\varepsilon$ |

#### Why not just train a classifier?

A natural first instinct is to treat this as standard supervised learning: we have a feature vector $(x, a)$ — the user's state plus which action was taken (one-hot encoded) — and a binary target (click = 1, no click = 0).  Train $\hat{r}(x, a) \approx p(\text{click} \mid x, a)$ and use it to pick the best action.

This is called the **direct method**, and it *almost* works — but breaks down once $\varepsilon < 1$.  Once the policy starts exploiting, action selection becomes **biased**: the model sends its current-best action to most users and rarely tries alternatives.  So we accumulate lots of training data for (high-purchase user, "Personalized picks") but very little for (high-purchase user, "See what's new").  The model becomes well-calibrated where it already exploits, but poorly calibrated for the actions it avoids — and it can never discover it's wrong because it rarely tries them.  This creates a **feedback loop**: biased data → biased model → more biased data.

#### How DR fixes this

On each round, VW needs a cost estimate for *every* action to do a proper gradient update — not just the one it chose.  First, a note on notation — VW works in **costs** (lower is better), not rewards:

| Symbol | Meaning |
|--------|---------|
| $c$ | **Observed cost** for the chosen action.  Click → $c = 0$ (good), no click → $c = 1$ (bad). |
| $\hat{c}_{\text{model}}(a') = \hat{W}[a'] \cdot x$ | VW's **predicted cost** for action $a'$ — a raw linear score trained on cost targets.  This is unbounded (can land outside [0, 1]) and uncalibrated; VW doesn't need proper probabilities, just correct *rankings*. |
| $\hat{c}_{\text{DR}}(a')$ | The **DR-corrected cost estimate** that VW actually uses for the gradient update. |

What matters for DR isn't the absolute scale of $\hat{c}_{\text{model}}$ — it's the **residual** $c - \hat{c}_{\text{model}}$: how far off the model was for the action we observed.

DR constructs $\hat{c}_{\text{DR}}$ differently depending on whether we observed the outcome:

**For the action we took** ($a' = a_{\text{chosen}}$):

$$
\hat{c}_{\text{DR}}(a') = \hat{c}_{\text{model}}(a') + \frac{c_{\text{observed}} - \hat{c}_{\text{model}}(a')}{\mu(a \mid x)}
$$

This starts with the model's prediction, then adds an IPS-corrected residual.  If the model was right ($c_{\text{observed}} \approx \hat{c}_{\text{model}}$), the correction is tiny.  If it was wrong, the correction is large — and scaled by $1/\mu$ to account for the probability of having chosen this action.

**For actions we didn't take** ($a' \neq a_{\text{chosen}}$):

$$
\hat{c}_{\text{DR}}(a') = \hat{c}_{\text{model}}(a')
$$

We have no observed outcome, so we fall back to the model's current prediction — it's all we have.

With these "filled in" costs for all $K$ actions, VW can update $\hat{W}$ as if it had full-information feedback.

#### Why "doubly robust"?

The overall DR value estimate combines both components:

$$
\hat{V}_{\text{DR}}(\pi) = \frac{1}{N} \sum_{i=1}^{N} \left[ \hat{r}(x_i, a_i) \;+\; \frac{\pi(a_i \mid x_i)}{\mu(a_i \mid x_i)} \bigl(r_i - \hat{r}(x_i, a_i)\bigr) \right]
$$

The name comes from a nice theoretical property: the estimate is consistent if *either* the model $\hat{r}$ is accurate *or* the propensities $\mu$ are accurate.  You only get in trouble if both are wrong simultaneously.

| Approach | What it does | Weakness |
|----------|-------------|----------|
| **Direct method** | Train $\hat{r}(x,a)$, update only on observed action | Selection bias — reinforces existing beliefs |
| **Pure IPS** | Weight observed cost by $1/\mu$, ignore model | High variance — especially when $\mu$ is small |
| **DR** | Use model as baseline, IPS-correct with observed data | Robust to errors in *either* component |

**IPS / SNIPS** are simpler batch estimators we use *after* the warm-up to compare candidate policies offline.  They don't require a learned model — just the logged rewards and propensities.

# 4. End-to-end process: how all the pieces fit together



Here's the full pipeline we run in this notebook, step by step:

---

**Phase 1 — Warm-up (pure exploration, $\varepsilon = 1$)**

> *Goal: collect unbiased training data for the model, and unbiased logs for OPE.*

1. Create a fresh VW model with $\varepsilon = 1$ (uniform random — every action gets probability $\frac{1}{K}$).
2. For each of $N$ rounds:
   - A dormant user arrives → observe context $x$
   - VW predicts action probabilities → all $0.25$ (uniform)
   - Sample an action $a$ at random, record propensity $\mu = 0.25$
   - Observe click/no-click → compute cost $= 1 - \text{click}$
   - Feed $(a, \text{cost}, \mu, x)$ back to VW → VW updates $\hat{W}$ internally (using Doubly Robust)
   - **Also** save $(x, a, \mu, \text{click})$ to a log for OPE later
3. Save the trained model to disk.

> **Why does learning during $\varepsilon = 1$ work?**  VW *is* updating $\hat{W}$ on every round, which would normally change the action probabilities.  But with $\varepsilon = 1$ the epsilon-greedy formula collapses to $\pi(a \mid x) = \frac{1}{K} = 0.25$ for all actions — the $\arg\max$ term gets zeroed out.  So the model learns quietly in the background, but the **exploration policy ignores what it learns**.  This gives us two nice properties: (1) the logged propensities are constant ($\mu = 0.25$, always exact, simplifying OPE), and (2) the saved model has reasonable $\hat{W}$ estimates ready for Phase 2.  If we used $\varepsilon < 1$ during warm-up, the changing $\hat{W}$ would shift action probabilities mid-collection, making propensities time-varying and OPE much harder.

*Output:* A warm-started model (has learned rough reward estimates from uniform data) **+** a batch of logged interactions.

---

**Phase 2 — Off-Policy Evaluation (OPE)**

> *Goal: pick the best $\varepsilon$ without deploying anything new.*

4. For each candidate $\varepsilon$ (e.g. $0.01, 0.05, 0.1, 0.2, 0.5$):
   - Load the warm-started model with that $\varepsilon$ → this gives a candidate policy $\pi_\varepsilon$
   - For every logged observation, compute the importance weight $w_i = \frac{\pi_\varepsilon(a_i \mid x_i)}{\mu(a_i \mid x_i)}$
   - Use IPS and SNIPS to estimate the CTR $\pi_\varepsilon$ *would have achieved*
5. Rank candidates by SNIPS score → pick the best $\varepsilon$.

*Output:* The $\varepsilon$ that OPE predicts will give the highest CTR (in our case, $\varepsilon = 0.01$).

---

**Phase 3 — Live deployment (exploit with minimal exploration)**

> *Goal: deploy the best policy and keep learning online.*

6. Load the warm-started model with the OPE-selected $\varepsilon$.
7. Run $N$ more rounds of the bandit loop — same 4 steps as warm-up, but now the policy *exploits* (puts ~99% probability on its best guess while still exploring 1%).
8. CTR should improve as the model continues learning and making better action choices.

*Output:* A policy that outperforms the uniform baseline — it has learned which message tone works best for which user segment.

---

> **When does online learning actually matter?**  During Phase 1 ($\varepsilon = 1$), online learning is a convenience, not a necessity — you could collect the logs with a coin flip and train a model on the batch afterward.  VW just happens to learn as it goes, giving us a warm-started model for free.  Online learning becomes *essential* in Phase 3 ($\varepsilon < 1$), where $\hat{W}$ directly controls which action the $\arg\max$ picks.  Each round's update immediately influences the next round's behavior, creating a virtuous cycle: better $\hat{W}$ → better actions → higher rewards → better $\hat{W}$.

---

```
 Phase 1: Warm-up          Phase 2: OPE              Phase 3: Live
┌─────────────────--┐    ┌────────────────-───┐    ┌──────────────────┐
│  ε = 1 (uniform)  │    │  Logged data from  │    │  ε = best (0.01) │
│                   │    │  Phase 1           │    │                  │
│  Collect data +   │-──▶│                    │-──▶│  Exploit + learn │
│  Train model      │    │  IPS / SNIPS eval  │    │  online          │
│  Save logs        │    │  Pick best ε       │    │                  │
└────────────────--─┘    └───────────────-────┘    └──────────────────┘
     ~9% CTR             (offline, no new          ~12% CTR
   (random baseline)       data needed)          (learned policy)
```